In [1]:
import pandas as pd
import numpy as np
import io
import os
from tqdm import tqdm
from curl_cffi import requests as cureq
from IPython.display import clear_output
import belo_horizonte_real_estate_market.functions.download_data as download_data
import belo_horizonte_real_estate_market.functions.sources as dicts

In [3]:
pd.set_option('display.max_columns', None)

In [2]:
list_resources = []
for key, value in tqdm(dicts.DATASETS["API_DATASETS_ID"].items()):
    print(f"\n{key}")
    df_resource = download_data.list_files(value)\
    .assign(dataset = key)

    list_resources.append(df_resource)
    clear_output(wait = True)

df_resources = pd.concat(objs = list_resources, ignore_index = True)

100%|██████████| 18/18 [00:04<00:00,  3.83it/s]


In [ ]:
df_itbi = df_resources\
.query("dataset == 'itbi' & format == 'CSV'")\
.apply(lambda df: download_data.get_csv_file(url = df["url"], dataset = "itbi", verbose = 0), axis = 1)

df_itbi = pd.concat(objs = list(df_itbi))\
.assign(area_terreno_total = lambda df: df["area_terreno_total"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(area_construida_adquirida = lambda df: df["area_construida_adquirida"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(area_adquirida_unidades_somadas = lambda df: df["area_adquirida_unidades_somadas"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(valor_declarado = lambda df: df["valor_declarado"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(valor_base_calculo = lambda df: df["valor_base_calculo"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(fracao_ideal_adquirida = lambda df: df["fracao_ideal_adquirida"].str.replace(",", ".").astype("float"))\
.assign(data_quitacao_transacao = lambda df: pd.to_datetime(df["data_quitacao_transacao"], format = "%d/%m/%Y"))\
.assign(ano_construcao_unidade = lambda df: [np.nan if i == 0 or i < 1800 and i > 2100 else i for i in df['ano_construcao_unidade']])\
.assign(tipo_construtivo_preponderante = lambda df: df["tipo_construtivo_preponderante"].map(dicts.DATASETS["TIPO_CONSTRUTIVO"]))\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))

In [6]:
df_itbi.to_parquet(path = "../data/raw_itbi.parquet", engine = "fastparquet", compression = "zstd")
df_itbi

,endereco,bairro,ano_construcao_unidade,area_terreno_total,area_construida_adquirida,area_adquirida_unidades_somadas,padrao_acabamento_unidade,fracao_ideal_adquirida,tipo_construtivo_preponderante,descricao_tipo_ocupacao_unidade,valor_declarado,valor_base_calculo,zona_uso_itbi,data_quitacao_transacao,urlfile
0,AVE AFONSO PENA 3924 - GARAGE 60 - CRUZEIRO - ...,CRUZEIRO,1976.0,1119.00,28.53,28.53,P3,0.004470,VAGA DE GARAGEM NAO RESIDENCIAL,NAO RESIDENCIAL,1000.00,11411.56,ZA,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
1,AVE AMAZONAS 718 - APT 704 - CENTRO - 30180-00...,CENTRO,1960.0,1030.00,126.99,126.99,P2,0.007197,APARTAMENTO,RESIDENCIAL,85000.00,85000.00,ZHIP,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
2,AVE AUGUSTO DE LIMA 1276 - APT 301 - BARRO PRE...,BARRO PRETO,1978.0,544.00,135.55,135.55,P3,0.025843,APARTAMENTO,RESIDENCIAL,121500.00,121500.00,ZCBH,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
3,AVE AUGUSTO DE LIMA 1276 - GARAGE 14 - BARRO P...,BARRO PRETO,1978.0,544.00,11.45,11.45,P3,0.002182,VAGA DE GARAGEM RESIDENCIAL,RESIDENCIAL,13500.00,13500.00,ZCBH,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
4,AVE AUGUSTO DE LIMA 233 - SALA 1439 - CENTRO -...,CENTRO,1967.0,4426.00,25.20,25.20,P3,0.000450,SALA,NAO RESIDENCIAL,9041.00,10354.49,ZHIP,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2233,RUA VARGINHA 463 - BLOCO A APT 1503 - COLEGIO ...,COLEGIO BATISTA,1983.0,1578.00,107.00,107.00,P3,0.012202,APARTAMENTO,RESIDENCIAL,219072.01,472512.00,ZAP,2025-11-28,pda_itbi_relatorio_202511.csv
2234,AVE BIAS FORTES 1577 - GARAGE 14 - BARRO PRETO...,BARRO PRETO,1983.0,375.00,12.00,12.00,P3,0.003043,VAGA DE GARAGEM RESIDENCIAL,RESIDENCIAL,26863.85,35078.40,ZCBH,2025-11-30,pda_itbi_relatorio_202511.csv
2235,RUA HENRIQUE GORCEIX 2120 - BLOCO V APT 403 - ...,JARDIM MONTANHES,1990.0,4755.29,70.70,70.70,P2,0.008929,APARTAMENTO,RESIDENCIAL,294000.00,294000.00,ZAR2,2025-11-30,pda_itbi_relatorio_202511.csv
2236,RUA MARIA HEILBUTH SURETTE 1312 - APT 1201 - B...,BURITIS,2025.0,3308.00,194.48,194.48,P4,0.017478,APARTAMENTO,RESIDENCIAL,1245016.12,1245016.12,ZAR2,2025-11-30,pda_itbi_relatorio_202511.csv


In [ ]:
df_resource = df_resources\
.query("dataset.str.contains('cadastro_imobiliario') & format == 'CSV'")\
.assign(year = lambda df: df["name"].apply(lambda x: x[:4]))\
.groupby(["year", "dataset"])\
.apply(lambda df: df.iloc[[-1]], include_groups = False)\
.reset_index()\
.astype({"year": "int"})\
.assign(aux_sep = lambda df: np.where(df["year"] < 2024, ",", ";"))\
.apply(lambda df: download_data.get_csv_file(url = df["url"], separate = df["aux_sep"], verbose = 0), axis = 1)


In [116]:
df_cadastro_imobiliario = pd.DataFrame()
for df in df_resource:
    df_cadastro_imobiliario = pd.concat(objs = [df_cadastro_imobiliario, df], ignore_index = True, axis = 0)\
    .drop_duplicates(["indice_cadastral", "nulotctm"])

df_cadastro_imobiliario = df_cadastro_imobiliario\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))\
.drop(columns = ["frequencia_coleta", "ind_meio_fio", "ind_pavimentacao", "ind_arborizacao", "ind_galeria_pluvial",
                 "ind_iluminacao_publica", "ind_rede_esgoto", "ind_rede_agua", "ind_rede_telefonica"])\
.astype(dtype = {"cep": "str", "numero_imovel": "str", "nulotctm": "str"})\
.assign(numero_imovel = lambda df: df["numero_imovel"].str.replace("\\.0", "", regex = True))\
.assign(cep = lambda df: df["cep"].str.replace("^0$", "", regex = True))

In [123]:
df_cadastro_imobiliario.to_parquet(path = "../data/raw_cadastro_imobiliario.parquet", engine = "fastparquet", compression = "zstd")
df_cadastro_imobiliario

,id_iptu_ctm,indice_cadastral,nulotctm,zoneamento_pviptu,area_terreno,area_construcao,tipo_construtivo,tipo_ocupacao,padrao_acabamento,quantidade_economias,fracao_ideal,tipo_logradouro,nome_logradouro,numero_imovel,cep,zona_homogenia,tipologia,geometria,urlfile
0,274,203156 015 001X,110776600280,ZAR2,360.00,132.00,CASA,RESIDENCIAL,P1,1.0,1.000000,RUA,BUENO DE RIVERA,331,30622060,BA312,DEMAIS CASOS,"POLYGON ((605511.7 7789921.5,605509.44 7789925...",20221201_regional_barreiro_cadastro_imobiliari...
1,275,204065 052 0010,120516800225,ZAR2,288.00,267.73,CASA,RESIDENCIAL,P3,1.0,1.000000,RUA,WILSON TAVARES RIBEIRO,527,30644260,BA217,FRENTE,"POLYGON ((602635.7 7789636,602632.25 7789645,6...",20221201_regional_barreiro_cadastro_imobiliari...
2,276,204073 301 0537,121370400175,ZAR2,13530.00,76.00,APARTAMENTO,RESIDENCIAL,P2,1.0,0.015152,RUA,ANTONIO TEIXEIRA DIAS,1755,30642270,BA217,FRENTE,"POLYGON ((602939.1 7789829,602928.94 7789837,6...",20221201_regional_barreiro_cadastro_imobiliari...
3,277,204073 301 0553,121370400175,ZAR2,13530.00,76.00,APARTAMENTO,RESIDENCIAL,P2,1.0,0.015152,RUA,ANTONIO TEIXEIRA DIAS,1755,30642270,BA217,FRENTE,"POLYGON ((602939.1 7789829,602928.94 7789837,6...",20221201_regional_barreiro_cadastro_imobiliari...
4,278,204073 301 0570,121370400175,ZAR2,13530.00,76.00,APARTAMENTO,RESIDENCIAL,P3,1.0,0.015152,RUA,ANTONIO TEIXEIRA DIAS,1775,30642270,BA217,FRENTE,"POLYGON ((602939.1 7789829,602928.94 7789837,6...",20221201_regional_barreiro_cadastro_imobiliari...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
983976,815939,988115 017 0032,210335000010,ZAR2,543.30,250.00,CASA,RESIDENCIAL,P2,2.0,0.500000,RUA,LUIZ CANTAGALLI,55,31578690,VN209,ALINHAMENTO,"POLYGON ((604545.25 7808694,604527.94 7808696....",20251103_regional_venda_nova_cadastro_imobilia...
984024,458451,971001 302 0015,210827801090,ZAP,402.45,23.56,BARRACAO,RESIDENCIAL,P2,2.0,1.000000,RUA,CRISANTO MUNIZ,462,31535450,VN306,FRENTE,"POLYGON ((606816.6 7809026,606796.8 7809041,60...",20251103_regional_venda_nova_cadastro_imobilia...
984137,820869,988099 005 0018,211043000405,ZAR2,792.00,454.00,CASA,RESIDENCIAL,P3,2.0,0.500000,RUA,PAULO EMILIO PINTO,125,31580060,VN216,FRENTE,"POLYGON ((604584 7807900.5,604589.3 7807908,60...",20251103_regional_venda_nova_cadastro_imobilia...
984268,732207,995007 021 0012,210354900320,ZAP,200.00,355.28,CASA,RESIDENCIAL,P3,2.0,1.000000,RUA,ANTONIO BICALHO LANA,201,31578200,VN207,ALINHAMENTO,"POLYGON ((604578.06 7809075.5,604560.3 7809084...",20251103_regional_venda_nova_cadastro_imobilia...


In [125]:
df_resource = df_resources\
.query("dataset == 'enderecamento' & format == 'CSV'")\
.reset_index()\
.apply(lambda df: download_data.get_csv_file(url = df["url"], verbose = 0), axis = 1)

In [131]:
df_enderecamento = pd.DataFrame()
for df in df_resource:
    df_enderecamento = pd.concat(objs = [df_enderecamento, df], ignore_index = True, axis = 0)\
    .drop_duplicates(["idend", "id_edc"])

df_enderecamento = df_enderecamento\
.astype(dtype = {"cep": "str", "numero_imovel": "str"})\
.assign(numero_imovel = lambda df: df["numero_imovel"].str.replace("\\.0", "", regex = True))\
.assign(cep = lambda df: df["cep"].str.replace("\\.0", "", regex = True))\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))

In [134]:
df_enderecamento.to_parquet(path = "../data/raw_enderecamento.parquet", engine = "fastparquet", compression = "zstd")
df_enderecamento

,idend,id_edc,id_logradouro,sigla_tipo_logradouro,desc_tipo_logradouro,nome_logradouro,numero_imovel,letra_imovel,id_bairro_popular,num_bairro_popular,nome_bairro_popular,id_bairro_oficial,num_bairro_oficial,tipo_bairro_oficial,nome_bairro_oficial,id_regional,nome_regional,cep,existencia_num_local,situacao_pbh,geometria,urlfile
0,09437100029,727545,94371,RUA,RUA,JOAQUIM HENRIQUES CARDOSO,29,NaN,120,738,Ouro Preto,51,304,Bairro,Ouro Preto,8,PAMPULHA,31320100,Sim,NaN,POINT (606171.65 7801411.08),20240102_endereco.csv
1,08064400450A,727546,80644,RUA,RUA,SERGIO MIRANDA MOREIRA,450,A,120,738,Ouro Preto,51,304,Bairro,Ouro Preto,8,PAMPULHA,31320060,Sim,NaN,POINT (606197.76 7801365.63),20240102_endereco.csv
2,09437100033B,727550,94371,RUA,RUA,JOAQUIM HENRIQUES CARDOSO,33,B,120,738,Ouro Preto,51,304,Bairro,Ouro Preto,8,PAMPULHA,31320100,Sim,NaN,POINT (606166.03 7801424.22),20240102_endereco.csv
3,09437100042,727551,94371,RUA,RUA,JOAQUIM HENRIQUES CARDOSO,42,NaN,120,738,Ouro Preto,51,304,Bairro,Ouro Preto,8,PAMPULHA,31320100,Sim,NaN,POINT (606161.40 7801412.18),20240102_endereco.csv
4,09571100561,727552,95711,AVE,AVENIDA,ANTONIO AUGUSTO DA SILVA,561,NaN,120,738,Ouro Preto,51,304,Bairro,Ouro Preto,8,PAMPULHA,31320070,NaN,Número Oficial,POINT (606151.84 7801421.89),20240102_endereco.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1519755,17105600105E,990191,171056,RUA,RUA,PAI JOAQUIM,105,E,19,624,Cabana do Pai Tomás,152,42,Vila,Cabana do Pai Tomás,7,OESTE,30512330,Sim,NaN,POINT (604588.94 7793989.54),20251103_endereco.csv
1519757,17105600100A,990189,171056,RUA,RUA,PAI JOAQUIM,100,A,19,624,Cabana do Pai Tomás,152,42,Vila,Cabana do Pai Tomás,7,OESTE,30512330,Sim,NaN,POINT (604566.10 7793983.01),20251103_endereco.csv
1520916,08293000569,990533,82930,RUA,RUA,ANTONIO FALABELA,569,NaN,151,775,Santa Terezinha,48,329,Bairro,Itatiaia,8,PAMPULHA,31360200,a confimar,Número Oficial,POINT (603943.38 7802170.29),20251103_endereco.csv
1520917,31249900006,990646,312499,BEC,BECO,DOZE DE OUTUBRO,6,NaN,123,741,Palmeiras,211,498,Bairro,Palmeiras,7,OESTE,30575722,Sim,NaN,POINT (605991.60 7790757.75),20251103_endereco.csv


In [4]:
df_resource = df_resources\
.query("dataset.str.contains('atividades_economicas') & format == 'CSV'")\
.query("dataset != 'atividades_economicas_autonomos'")\
.reset_index()\
.assign(year_month = lambda df: df["name"].apply(lambda x: x[:6]))\
.assign(aux_sep = lambda df: np.where(df["year_month"] < "202308", ",", ";"))\
.apply(lambda df: download_data.get_csv_file(url = df["url"], separate = df["aux_sep"], verbose = 1), axis = 1)

G:\Meu Drive\belo_horizonte_real_estate_market\belo_horizonte_real_estate_market\functions\download_data.py:47: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data, sep = separate, encoding = 'utf-8')


https://ckan.pbh.gov.br/dataset/3449df83-944b-4672-835d-d3e4a7bf7f48/resource/02c1f2c0-1dfb-4204-94d1-2b229187e0ba/download/20220601_atividade_economica.csv: successful downloaded data!


G:\Meu Drive\belo_horizonte_real_estate_market\belo_horizonte_real_estate_market\functions\download_data.py:47: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data, sep = separate, encoding = 'utf-8')


https://ckan.pbh.gov.br/dataset/3449df83-944b-4672-835d-d3e4a7bf7f48/resource/c504cb52-7455-47e2-8863-4d935b7d3270/download/20220701_atividade_economica.csv: successful downloaded data!


G:\Meu Drive\belo_horizonte_real_estate_market\belo_horizonte_real_estate_market\functions\download_data.py:47: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data, sep = separate, encoding = 'utf-8')


...

https://ckan.pbh.gov.br/dataset/3449df83-944b-4672-835d-d3e4a7bf7f48/resource/d2262e21-b3dc-481a-a7de-86f54b06bb63/download/20240701_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/3449df83-944b-4672-835d-d3e4a7bf7f48/resource/58700a42-7f7b-496b-b077-b5b4143aeee0/download/20240801_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/e1b511d5-0a17-4b2b-9392-5a02525d0e63/download/20240926_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/b3b25b26-8fd9-4f90-8962-69d806731ead/download/20241001_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/1d08b8ca-5d8a-4c4b-a7fb-5ae04f47f485/download/20241104_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/r

G:\Meu Drive\belo_horizonte_real_estate_market\belo_horizonte_real_estate_market\functions\download_data.py:47: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data, sep = separate, encoding = 'utf-8')


https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/216bb791-76c4-4e16-bb36-fef15af27c25/download/20250306_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/d5090246-039f-4890-baf7-26ba2a426d9e/download/20250401_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/968cc372-fc1a-4418-b3d9-af9b64bc968c/download/20250505_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/f8dac429-d6f9-4647-afc8-11f38272a2a9/download/20250602_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/2a68f929-1d3a-459b-b0e0-8d1ea2f154c0/download/20250701_atividade_economica.csv: successful downloaded data!


G:\Meu Drive\belo_horizonte_real_estate_market\belo_horizonte_real_estate_market\functions\download_data.py:47: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data, sep = separate, encoding = 'utf-8')


https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/d32dd644-1fd0-4f03-b8fe-e02c97d3ef31/download/20250801_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/5ad4867a-0121-4068-a172-acf153593a82/download/20250901_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/93aa8da7-c81f-4e05-aabe-e43fee4a2d47/download/20251001_atividade_economica.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/33e9dcb1-126f-4cde-8c80-d1927a965430/resource/f7a5b519-0f85-4fc3-af92-e1c71329fd04/download/20251103_atividade_economica.csv: successful downloaded data!


In [10]:
??download_data.get_csv_file

Signature: download_data.get_csv_file(url, dataset=None, separate=';', verbose=1)
Docstring: <no docstring>
Source:   
def get_csv_file(url, dataset = None, separate = ";", verbose = 1):
    if url == "":
        return None

    # headers = {
    #     "User-Agent": "Mozilla/5.0",
    #     "Referer": "https://ckan.pbh.gov.br"
    # }

    # scraper = cloudscraper.create_scraper()
    # response = scraper.get(url, headers = headers)
    response = cureq.get(url, impersonate = "chrome", timeout = 300)

    if response.status_code == 200:
        # Get the binary content
        csv_data = io.BytesIO(response.content)
        
        # read the csv file using pandas
        df = pd.DataFrame()
        try:
            df = pd.read_csv(csv_data, sep = separate, encoding = 'utf-8')
            if verbose: print(f"{url}: successful downloaded data!")
        except Exception as e:
            # Try another encoding if the first fail
            csv_data.seek(0)
            df = pd.read_cs

,id_atvecon,cnae_principal,descricao_cnae,cnae_secundarias,natureza_juridica,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,nome,nome_fantasia,cnpj,data_inicio_atividade,nome_regional,geometria,urlfile
71,2045,4520001.0,SERVICOS DE MANUTENCAO E REPARACAO MECANICA DE...,"4520002, 4520003, 4520004",SOCIEDADE EMPRESÁRIA LIMITADA,DEMAIS,85.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,CARLOS GOMES,402,NaN,SANTO ANTONIO,OFICINA MECANICA JULIO MURTA LTDA,NaN,77367000103,01/05/1994,CENTRO-SUL,POINT (610093.51 7794490.31),https://ckan.pbh.gov.br/dataset/3449df83-944b-...
94,2068,9319101.0,PRODUCAO E PROMOCAO DE EVENTOS ESPORTIVOS,"7112000, 7220700, 7490199, 8230001, 9002701, 9...",SOCIEDADE SIMPLES LIMITADA,MICROEMPRESA - ME,9.0,N,N,ESCRITÓRIO ADMINSTRATIVO,ESTABELECIMENTO FIXO,RUA,CONGONHAS,648,NaN,SANTO ANTONIO,CRC CONSULTORIA EM ENGENHARIA E TREINAMENTO LTDA,NaN,73229000156,05/12/1995,CENTRO-SUL,POINT (610765.94 7794456.82),https://ckan.pbh.gov.br/dataset/3449df83-944b-...
110,2084,8511200.0,EDUCACAO INFANTIL-CRECHE,NaN,ASSOCIAÇÃO PRIVADA,DEMAIS,156.0,N,N,UNIDADE PRODUTIVA,ATIVIDADES DESENVOLVIDAS FORA DO ESTABELECIMENTO,RUA,TENENTE MARINO FREIRE,157,NaN,MARIA HELENA,CRECHE DO CONSELHO COMUNITARIO INTEGRACAO DE V...,NaN,77616000160,11/11/1984,VENDA NOVA,POINT (605500.13 7810106.62),https://ckan.pbh.gov.br/dataset/3449df83-944b-...
175,883,4530701.0,COMERCIO POR ATACADO DE PECAS E ACESSORIOS NOV...,4530703,SOCIEDADE EMPRESÁRIA LIMITADA,DEMAIS,1500.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,VILA RICA,688,NaN,PADRE EUSTAQUIO,BY CAR COMERCIO E REPRESENTACOES LTDA,BY CAR COMPONENTES,86783000178,06/06/1994,NOROESTE,POINT (606685.97 7797524.29),https://ckan.pbh.gov.br/dataset/3449df83-944b-...
180,888,4120400.0,CONSTRUCAO DE EDIFICIOS,4299501,SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,50.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,DOS TUPINAMBAS,190,SALA 5,CENTRO,ARAUCHE E ARAUCHE EMPREENDIMENTOS LTDA,ARAUCHE E ARAUCHE EMPREENDIMENTOS LTDA,76605000166,02/05/1994,CENTRO-SUL,POINT (611354.72 7797204.14),https://ckan.pbh.gov.br/dataset/3449df83-944b-...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496615,494859,8112500.0,CONDOMINIOS PREDIAIS,NaN,CONDOMÍNIO EDILÍCIO,DEMAIS,10.0,N,N,UNIDADE PRODUTIVA,NaN,RUA,FELIPE DOS SANTOS,344,APTO 302,LOURDES,CONDOMINIO DO EDIFICIO MARIA CECILIA,NaN,67922000116,13/05/1994,CENTRO-SUL,POINT (610422.45 7795432.69),https://ckan.pbh.gov.br/dataset/3449df83-944b-...
496630,493384,8112500.0,CONDOMINIOS PREDIAIS,NaN,CONDOMÍNIO EDILÍCIO,DEMAIS,10.0,N,N,UNIDADE PRODUTIVA,NaN,RUA,CELIA DE SOUZA,322,NaN,SAGRADA FAMILIA,CONDOMINIO EDIFICIO PORTAL 1,NaN,85020000102,09/05/1994,LESTE,POINT (612231.63 7798660.00),https://ckan.pbh.gov.br/dataset/3449df83-944b-...
496996,493425,8112500.0,CONDOMINIOS PREDIAIS,NaN,CONDOMÍNIO EDILÍCIO,DEMAIS,10.0,N,N,UNIDADE PRODUTIVA,NaN,RUA,MARTIM FRANCISCO,582,NaN,GUTIERREZ,CONDOMINIO DO EDIFICIO CARAIBAS,NaN,92355000158,16/06/1994,OESTE,POINT (608769.83 7794996.20),https://ckan.pbh.gov.br/dataset/3449df83-944b-...
497301,493101,4391600.0,OBRAS DE FUNDACOES,NaN,SOCIEDADE EMPRESÁRIA LIMITADA,DEMAIS,50.0,S,N,ESCRITÓRIO ADMINSTRATIVO,NaN,AVE,RAJA GABAGLIA,1001,SALA: 404 E 405;,LUXEMBURGO,TECNICAS EM GEOTECNIA LTDA,NaN,56013000182,01/04/1994,CENTRO-SUL,POINT (609187.01 7794360.96),https://ckan.pbh.gov.br/dataset/3449df83-944b-...


In [19]:
df_atividades_economicas = pd.DataFrame()
for df in df_resource:
    df = df.astype({"cnpj": "str"}).query("~cnpj.str.contains('\+')")
    df_atividades_economicas = pd.concat(objs = [df_atividades_economicas, df], ignore_index = True, axis = 0)\
    .drop_duplicates(["cnpj", "nome", "desc_logradouro", "nome_logradouro", "numero_imovel", "complemento", "nome_bairro"])

df_atividades_economicas = df_atividades_economicas\
.astype({"cnpj": "str"})\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))\
.assign(descricao_cnae = lambda df: df["descricao_cnae"].fillna(df["descricao_cnae_principal"]))\
.assign(cnae_secundarias = lambda df: df["cnae_secundarias"].fillna(df["cnae"]))\
.drop(columns = ["descricao_cnae_principal", "cnae", "id_atvecon", "id_ativ_econ_estabelecimento"])\
.astype("str")

In [21]:
df_atividades_economicas.to_parquet(path = "../data/raw_atividades_economicas.parquet", engine = "fastparquet", compression = "zstd")
df_atividades_economicas

,cnae_principal,descricao_cnae,cnae_secundarias,natureza_juridica,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,nome,nome_fantasia,cnpj,data_inicio_atividade,nome_regional,geometria,urlfile,ind_possui_alvara
0,8711502.0,INSTITUICOES DE LONGA PERMANENCIA PARA IDOSOS,nan,SOCIEDADE SIMPLES LIMITADA,MICROEMPRESA - ME,300.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,ISMAIL VILELA LIMA,137,nan,BRAUNAS,HOTEL RESIDENCIA CADEIRA DE BALANCO LTDA,CADEIRA DE BALANCO,70947296000138,01/04/1993,PAMPULHA,POINT (604031.633099028 7805359.56268045),20220601_atividade_economica.csv,nan
1,4672900.0,COMERCIO ATACADISTA DE FERRAGENS E FERRAMENTAS,4744001,SOCIEDADE EMPRESÁRIA LIMITADA,DEMAIS,2130.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,JOSE ALVES DA SILVA,80,nan,CAICARAS,ITALY LINE FERRAGENS LTDA,nan,71032189000142,05/04/1993,NOROESTE,POINT (608127.253256548 7799690.5865841),20220601_atividade_economica.csv,nan
2,8599604.0,TREINAMENTO EM DESENVOLVIMENTO PROFISSIONAL E ...,"7020400, 7490104",SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,40.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVE,GETULIO VARGAS,254,SALA 903,FUNCIONARIOS,DESEMPENHO RESPONSAVEL LTDA,nan,42778522000169,01/10/1992,CENTRO-SUL,POINT (612339.582296524 7795585.67483887),20220601_atividade_economica.csv,nan
3,7311400.0,VEICULACAO E DIVULGACAO DE PROPAGANDA E PUBLIC...,6821801,SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,9.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,ALVARENGA PEIXOTO,411,APT: 102;,LOURDES,NEXO COMUNICACAO LTDA,nan,71053524000199,01/06/1993,CENTRO-SUL,POINT (610660.612125093 7795728.62675115),20220601_atividade_economica.csv,nan
4,7119704.0,SERVICOS DE PERICIA TECNICA RELACIONADOS A SEG...,7112000,SOCIEDADE SIMPLES LIMITADA,DEMAIS,119.0,N,N,UNIDADE PRODUTIVA,ATIVIDADES DESENVOLVIDAS FORA DO ESTABELECIMENTO,RUA,SERGIPE,1167,SALA 701,SAVASSI,CROENGE LTDA,CLAUDIO ROCHA ESCRITORIO DE PERICIAS,70952007000199,30/04/1993,CENTRO-SUL,POINT (611162.819664811 7795161.82901748),20220601_atividade_economica.csv,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1538205,6204000.0,CONSULTORIA EM TECNOLOGIA DA INFORMACAO,6204000,SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,10.0,S,N,ESCRITÓRIO ADMINSTRATIVO,nan,RUA,GUILHERME DE ALMEIDA,43,APT 402,SANTO ANTONIO,DUANI TEC LTDA,DUANI TEC,59201283000182,30-01-2025,nan,POINT (610127.93 7793904.97),20251103_atividade_economica.csv,SIM
1538500,2539001.0,"SERVIÇOS DE USINAGEM, SOLDA E TORNEARIA","2539001, 8211300, 8299799",SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,40.0,S,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,CLAUDIO,62,SALA: F,PRADO,MECAL MECANICA E PRESTACAO DE SERVICOS LTDA,MECAL MECANICA CALDEIRARIA LTDA,45936785000147,29-01-2025,nan,POINT (608936.68 7797012.74),20251103_atividade_economica.csv,SIM
1538601,8211300.0,SERVICOS COMBINADOS DE ESCRITORIO E APOIO ADMI...,"7020400, 7319002, 7490199, 8211300, 8219999, 8...",SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,18.0,S,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,HIDRA,301,SALA: 501,SANTA LUCIA,SALUM SOLUCOES AMBIENTAIS LTDA,LAS SOLUCOES AMBIENTAIS,59286334000116,04-02-2025,nan,POINT (610187.88 7792329.11),20251103_atividade_economica.csv,SIM
1538609,8230001.0,"SERVICOS DE ORGANIZACAO DE FEIRAS, CONGRESSOS,...",8230001,EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,"PORTA A PORTA, POSTOS MÓVEIS OU POR AMBULANTES",RUA,AMPARO DA SERRA,89,CASA,ARAGUAIA,59.282.548 PATRICIA DANIELLE NUNES FERREIRA DE...,nan,59282548000114,04-02-2025,nan,POINT (603962.17 7789312.37),20251103_atividade_economica.csv,NÃO


In [8]:
df_resource = df_resources\
.query("dataset == 'atividades_economicas_autonomos' & format == 'CSV'")\
.assign(year_month = lambda df: df["name"].apply(lambda x: x[:6]))\
.assign(aux_sep = lambda df: np.where(df["year_month"] < "202308", ",", ";"))\
.apply(lambda df: download_data.get_csv_file(url = df["url"], separate = df["aux_sep"], verbose = 1), axis = 1)

https://ckan.pbh.gov.br/dataset/0e6890a3-6957-4d54-b6a6-0c77b97b1b88/resource/e6780cb5-7ae3-4f0c-913f-55c09f4561d1/download/20220601_atividade_economica_autonomos.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/0e6890a3-6957-4d54-b6a6-0c77b97b1b88/resource/c8c5d1b5-a5d2-4af6-a469-ca612968f212/download/20220701_atividade_economica_autonomo.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/0e6890a3-6957-4d54-b6a6-0c77b97b1b88/resource/13deb25c-c581-4fe1-8401-dc726dbb9d55/download/20220801_atividade_economica_autonomo.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/0e6890a3-6957-4d54-b6a6-0c77b97b1b88/resource/f5bda6e5-d91b-437e-b61f-a18cea63d644/download/20220908_atividade_economica_autonomo.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/0e6890a3-6957-4d54-b6a6-0c77b97b1b88/resource/7e209a60-ee10-4224-bf02-7f99f93b9f93/download/20221003_atividade_economica_autonomo.csv: successful downloaded data!
https://ckan.pbh.gov.br/

In [ ]:
df_atividades_economicas_autonomos = pd.DataFrame()
for df in df_resource:
    df_atividades_economicas_autonomos = pd.concat(objs = [df_atividades_economicas_autonomos, df], ignore_index = True, axis = 0)\
    .drop_duplicates(["cod_inscricao_municipal", "logradouro", "numero"])

df_atividades_economicas_autonomos = df_atividades_economicas_autonomos\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))

In [43]:
df_atividades_economicas_autonomos.to_parquet(path = "../data/raw_atividades_economicas_autonomos.parquet", engine = "fastparquet", compression = "zstd")
df_atividades_economicas_autonomos

,cod_inscricao_municipal,area_utilizada,area_sujeita_tfs,codigo_cbo,descricao_cbo,tipo_logradouro,logradouro,numero,complemento,data_inicio_atividade,nome_bairro_popular,nome_regional,geometria,urlfile
0,0590269001X,10,0.0,354505,CORRETOR DE SEGUROS,RUA,CANTOR LUIZ GONZAGA,485.0,APTO 203,02/01/1992,Castelo,PAMPULHA,POINT (604339.165887895 7800695.22328009),20220601_atividade_economica_autonomos.csv
1,1111208001X,10,NaN,252210,CONTADOR,RUA,DOUTOR CRISTIANO REZENDE,2882.0,CASA CASA A,18/10/2018,Araguaia,BARREIRO,POINT (604586.426949785 7789100.47520631),20220601_atividade_economica_autonomos.csv
2,10170340012,10,10.0,225170,MÉDICO GENERALISTA,RUA,ESMERALDO BOTELHO,216.0,APT 203,23/02/2017,Buritis,OESTE,POINT (608049.288798734 7790433.08092281),20220601_atividade_economica_autonomos.csv
3,06600490024,17,17.0,223208,CIRURGIAO DENTISTA - CLINICO GERAL,RUA,CONCEICAO DO MATO DENTRO,445.0,NaN,10/08/2008,Ouro Preto,PAMPULHA,POINT (606689.475387283 7802416.31184928),20220601_atividade_economica_autonomos.csv
4,06625190016,41,41.0,251510,PSICOLOGO CLINICO,RUA,DESEMBARGADOR JORGE FONTANA,428.0,SALA 1008,22/08/2007,Belvedere,CENTRO-SUL,POINT (610491.10892488 7790852.82317423),20220601_atividade_economica_autonomos.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30579,06768100010,10,10.0,322130,ESTETICISTA,RUA,RADIALISTA JUSCELINO SOUZA,297.0,,15-02-2012,Céu Azul,VENDA NOVA,POINT (603980.62 7808622.20),20250204_atividade_economica_autonomo.csv
30580,1485966001X,20,20.0,223605,FISIOTERAPEUTA,PRACA,CHICO PEREIRA,90.0,NaN,06-07-2023,Dom Cabral,NOROESTE,POINT (604701.33 7796741.14),20250204_atividade_economica_autonomo.csv
30581,05762240017,10,10.0,516110,CABELEIREIRO,RUA,ARTUR DE SA,2229.0,LOJA,21-03-1988,União,NORDESTE,POINT (613178.69 7801540.99),20250204_atividade_economica_autonomo.csv
30582,12530490017,10,NaN,212315,ADMINISTRADOR DE SISTEMAS OPERACIONAIS,RUA,JOAO ARANTES,182.0,APT 302,18-09-2020,Cidade Nova,NORDESTE,POINT (612757.21 7800788.21),20250204_atividade_economica_autonomo.csv


In [32]:
df_resource = df_resources\
.query("dataset == 'edificacoes_licenciadas' & format == 'CSV'")\
.apply(lambda df: download_data.get_csv_file(url = df["url"], verbose = 1), axis = 1)

https://ckan.pbh.gov.br/dataset/602b0331-286a-4a59-a987-6f6f38f6ebec/resource/8ac7c58c-3a9c-474a-bca4-c67ca4c21895/download/20241003_projeto_edificacao_licenciado.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/602b0331-286a-4a59-a987-6f6f38f6ebec/resource/995d65d6-fb75-4fad-8588-2a494f5d5584/download/20241104_projeto_edificacao_licenciado.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/602b0331-286a-4a59-a987-6f6f38f6ebec/resource/a81c1f13-13b0-43f5-8985-07be5b71d179/download/20241129_projeto_edificacao_licenciado.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/602b0331-286a-4a59-a987-6f6f38f6ebec/resource/9195a272-1b48-415d-ad87-512cb870f072/download/20250102_projeto_edificacao_licenciado.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/602b0331-286a-4a59-a987-6f6f38f6ebec/resource/f8367d3a-c2a3-4dc2-8dd1-c0cb41600451/download/20250204_projeto_edificacao_licenciado.csv: successful downloaded data!
https://ckan.pbh.gov

In [ ]:
df_projetos_edificacao_licenciados = pd.DataFrame()
for df in df_resource:
    df_projetos_edificacao_licenciados = pd.concat(objs = [df_projetos_edificacao_licenciados, df], ignore_index = True, axis = 0)\
    .drop_duplicates(["id_projeto_edificacoes", "numero_processo"])

df_projetos_edificacao_licenciados = df_projetos_edificacao_licenciados\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))

In [47]:
df_projetos_edificacao_licenciados.to_parquet(path = "../data/raw_projetos_edificacao_licenciados.parquet", engine = "fastparquet", compression = "zstd")
df_projetos_edificacao_licenciados

,id_projeto_edificacoes,numero_processo,num_requerimento,situacao_requerimento,titulo_projeto,tipo,situacao_projeto,num_ultimo_alvara,dt_emissao_alvara_construcao,dt_concessao_ultimo_alvara,dt_validade_ultimo_alvara,data_comunicado_inicio_obra,data_ultima_baixa,tipo_ultima_baixa,endereco,lote_projeto,uso_geral,qtd_und_residencial,qtd_und_nao_residencial,area_construida,tipo_aprovacao,data_aprovacao,area_liquida,qtde_pavimentos,geometria,urlfile
0,258616,1.001523e+10,2021A06965,Deferido,LEVANTAMENTO DO ACRESCIMO,REGULARIZACAO,APROVADO,199200026.0,2021-12-09 20:36:28,1992-01-07 00:00:00,1992-02-21 00:00:00,NaN,1992-02-21 12:00:00,BAIXA TOTAL,NaN,"Zona Fiscal 453 Quarteirão 018A Lote(s) 001,00...",NÃO RESIDENCIAL,0,3,1005.48,Convencional,07-01-1992,0.00,2,"MULTIPOLYGON (((615258.14 7798375.31, 615264.4...",20241003_projeto_edificacao_licenciado.csv
1,266649,1.001547e+10,2022A09568,Deferido,APROVACAO INICIAL,LICENCIAMENTO,APROVADO,NaN,NaN,NaN,NaN,NaN,2002-07-18 12:00:00,BAIXA TOTAL,NaN,"Zona Fiscal 618 Quarteirão 155 Lote(s) 034,035",NÃO RESIDENCIAL,0,1,23780.81,Convencional,03-01-1995,0.00,4,"MULTIPOLYGON (((602445.13 7787300.54, 602446.1...",20241003_projeto_edificacao_licenciado.csv
2,125362,1.001268e+10,1994M01576,Deferido,MODIFICACAO COM ACRESCIMO DE AREA CONSTRUIDA,LICENCIAMENTO,APROVADO,199407693.0,1994-05-26 00:00:00,1992-01-09 00:00:00,1995-07-09 00:00:00,NaN,NaN,NaN,"AVE DEPUTADO ULTIMO DE CARVALHO, 605","Zona Fiscal 947 Quarteirão 100 Lote(s) 007,009",USO INVÁLIDO: PROJETO MIGRADO EM 2013,0,0,0.00,NaN,09-01-1992,531.69,3,"MULTIPOLYGON (((610702.73 7806331.22, 610695.2...",20241003_projeto_edificacao_licenciado.csv
3,299027,1.001214e+10,2024A21804,Deferido,LEVANTAMENTO DO ACRESCIMO,REGULARIZACAO,APROVADO,NaN,NaN,NaN,NaN,NaN,1998-04-03 12:00:00,BAIXA TOTAL,NaN,Zona Fiscal 102 Quarteirão 030 Lote(s) 023,RESIDENCIAL,6,0,743.27,Convencional,19-03-1998,0.00,7,"MULTIPOLYGON (((610950.99 7794084.67, 610940.9...",20241003_projeto_edificacao_licenciado.csv
4,233067,1.001215e+10,2019A04172,Deferido,APROVACAO INICIAL,LICENCIAMENTO,APROVADO,NaN,NaN,NaN,NaN,NaN,1992-07-01 12:00:00,BAIXA TOTAL,NaN,Zona Fiscal 313 Quarteirão 038 Lote(s) 018,RESIDENCIAL,1,0,243.21,Convencional,27-12-1989,0.00,2,"MULTIPOLYGON (((610352.22 7803335.49, 610325.2...",20241003_projeto_edificacao_licenciado.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166133,321729,NaN,2025R28355,Enviado,APROVACAO INICIAL,LICENCIAMENTO,REQUERIMENTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zona Fiscal 807 Quarteirão 111 Lote(s) 031,RESIDENCIAL,5,0,373.96,Convencional,NaN,344.04,4,"MULTIPOLYGON (((604997.80 7799116.08, 605008.3...",20251103_projeto_edificacao_licenciado.csv
166134,321730,NaN,2025A28358,Deferido,LEVANTAMENTO TOTAL,REGULARIZACAO,APROVADO,NaN,NaN,NaN,NaN,NaN,1981-08-07 12:00:00,BAIXA TOTAL,NaN,Zona Fiscal 947 Quarteirão 053 Lote(s) 013,RESIDENCIAL,2,0,186.56,Convencional,24-06-1981,0.00,1,"MULTIPOLYGON (((610876.77 7806302.10, 610863.9...",20251103_projeto_edificacao_licenciado.csv
166135,321731,NaN,2025R28359,Enviado,APROVACAO INICIAL,LICENCIAMENTO,REQUERIMENTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zona Fiscal 425 Quarteirão 116 Lote(s) 002A,NÃO RESIDENCIAL,0,1,248.90,Alvara na Hora,NaN,292.82,1,"MULTIPOLYGON (((612728.59 7799227.15, 612740.4...",20251103_projeto_edificacao_licenciado.csv
166136,321732,NaN,2025R28360,Enviado,LEVANTAMENTO DO ACRESC. C- PROJ DEMODIF.,REGULARIZACAO,REQUERIMENTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zona Fiscal 120 Quarteirão 036 Lote(s) 011,RESIDENCIAL,7,0,1402.10,Convencional,NaN,1054.65,4,"MULTIPOLYGON (((609888.85 7794233.99, 609891.0...",20251103_projeto_edificacao_licenciado.csv


In [61]:
df_resource = df_resources\
.query("dataset == 'qtd_lancamentos_iptu_bairro' & format == 'CSV'")\
.apply(lambda df: download_data.get_csv_file(url = df["url"], verbose = 1).assign(urlfile = df["name"]), axis = 1)

df_lancamentos_iptu_bairro = pd.concat(objs = list(df_resource), ignore_index = True)\
.sort_values(["bairro", "urlfile"])\
.assign(qde_lancamentos = lambda df: df["qde_lancamentos"].fillna(df["total_imoveis_sumarizado"]))\
.assign(valor_total_lancado = lambda df: df["valor_total_lancado"].fillna(df["valor_lancado_sumarizado"]))\
.drop(columns = ["total_imoveis_sumarizado", "valor_lancado_sumarizado"])

https://ckan.pbh.gov.br/dataset/177e1466-187d-47df-8638-1d02a060afc6/resource/412eddd3-f057-4716-a33f-61cc9c79dd38/download/iptu-bairro.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/177e1466-187d-47df-8638-1d02a060afc6/resource/9c271b60-e720-4fd8-b88e-0d9254a4da50/download/iptu-bairro.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/177e1466-187d-47df-8638-1d02a060afc6/resource/98a55e31-cfd4-4889-b659-f6ab3f580765/download/iptu-bairro-2023.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/177e1466-187d-47df-8638-1d02a060afc6/resource/da983867-0b51-4698-beb1-7a8a8481848a/download/iptu_bairro_2024.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/177e1466-187d-47df-8638-1d02a060afc6/resource/1cf22dd3-3d61-45a2-90ca-94f17c9e63b8/download/iptu_bairro_2025.csv: successful downloaded data!


In [62]:
df_lancamentos_iptu_bairro.to_parquet(path = "../data/raw_qtd_lancamentos_iptu_por_bairro.parquet", engine = "fastparquet", compression = "zstd")
df_lancamentos_iptu_bairro

,bairro,qde_lancamentos,valor_total_lancado,urlfile
216,AARAO REIS,622.0,"1.662.055,42",2021 - IPTU Bairro
668,AARAO REIS,616.0,"1.724.737,03",2022 - IPTU Bairro
1102,AARAO REIS,616.0,"1.801.835,03",2023 - IPTU Bairro
1359,AARAO REIS,622.0,"1.913.886,37",2024 - IPTU Bairro
1819,AARAO REIS,625.0,"2.026.568,73",2025 - IPTU Bairro
...,...,...,...,...
1295,ZILAH SPOSITO,10.0,"5.908,78",2023 - IPTU Bairro
1817,ZILAH SPOSITO,11.0,"7.558,41",2024 - IPTU Bairro
2278,ZILAH SPOSITO,11.0,"7.944,16",2025 - IPTU Bairro
1818,NaN,4.0,"5.756,98",2024 - IPTU Bairro


In [80]:
df_resource = df_resources\
.query("dataset == 'baixa_construcoes' & format == 'CSV'")\
.apply(lambda df: download_data.get_csv_file(url = df["url"], verbose = 1).assign(urlfile = df["name"]), axis = 1)

df_baixa_construcoes = pd.concat(objs = list(df_resource), ignore_index = True)\
.astype("string")

https://ckan.pbh.gov.br/dataset/aa138ff1-7229-4101-ba1c-399eee7de8be/resource/2d691fd2-bc0b-4ff9-b6c5-150edd7de22c/download/2019-01_baixas_construcao_concedidas.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/aa138ff1-7229-4101-ba1c-399eee7de8be/resource/8b7283ff-b382-44f5-938c-cf75cff38ea0/download/2019-02_baixas_construcao_concedidas.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/aa138ff1-7229-4101-ba1c-399eee7de8be/resource/283bf440-001b-406c-b340-2c06e973012e/download/2019-03_baixas_construcao_concedidas.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/aa138ff1-7229-4101-ba1c-399eee7de8be/resource/8f01e506-65de-40f3-9f0f-50499e5897da/download/2019-04_baixas_construcao_concedidas.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/aa138ff1-7229-4101-ba1c-399eee7de8be/resource/2c6ea794-a432-4284-8b18-db0fa09389d4/download/2019-05_baixas_construcao_concedidas.csv: successful downloaded data!
https://ckan.pbh.gov.br/datase

In [81]:
df_baixa_construcoes.to_parquet(path = "../data/raw_baixa_construcoes.parquet", engine = "fastparquet", compression = "zstd")
df_baixa_construcoes

,numero_do_processo,tipo_do_projeto,titulo_projeto,endereco,nome_bairro,zona_fiscal,quarteirao,lotes,area_terreno,area_construida,area_liquida,pavimentos,uso_do_projeto,und._residenciais,und._nao_residenciais,resp._tecnico_pela_execucao_da_obra,registro_profissional,data_concessao_da_baixa,tipo_da_baixa,urlfile
0,01-003371/13-02,LICENCIAMENTO,APROVACAO INICIAL,"RUA ARAPARI, s/n",SAO GERALDO,448.0,53,12,"420,00","1059,60","627,81",6.0,RESIDENCIAL,8.0,0.0,RICARDO STARLING DE MATOS,MG50501/D,25/01/2019,BAIXA TOTAL EM 2019-01-25,2019-01 - Baixas de Construção Concedidas
1,01-092459/16-06,LICENCIAMENTO,APROVACAO INICIAL,"ALA DOS OITIS, 247",SÃO LUÍZ,374.0,7,28,"1000,00","332,81","332,81",1.0,RESIDENCIAL,1.0,0.0,NATHALIA DUARTE PEREIRA ALVES,174164D,25/01/2019,BAIXA TOTAL EM 2019-01-25,2019-01 - Baixas de Construção Concedidas
2,01-088641/15-37,REGULARIZACAO,LEVANTAMENTO TOTAL,"RUA PAULO DO COUTO E SILVA, 155,157",HELIÓPOLIS,312.0,018A,14,"360,00","653,00","404,68",3.0,MISTO,1.0,1.0,<NA>,<NA>,14/01/2019,BAIXA TOTAL EM 2019-01-14,2019-01 - Baixas de Construção Concedidas
3,01-035389/16-62,REGULARIZACAO,LEVANTAMENTO DO ACRESCIMO,"RUA COLONITA, 23,29",ITATIAIA,329.0,42,17,"360,00","414,63","361,63",3.0,RESIDENCIAL,1.0,0.0,<NA>,<NA>,22/01/2019,BAIXA TOTAL EM 2019-01-22,2019-01 - Baixas de Construção Concedidas
4,01-107121/17-27,LICENCIAMENTO,APROVACAO INICIAL,"RUA JOSIAS CASIMIRO, 176",SÃO JOÃO,427.0,37,9,"433,04","463,39","420,91",3.0,RESIDENCIAL,9.0,0.0,JOSE RICARDO BENJAMIM DE ANDRADE,80679-0,24/01/2019,BAIXA TOTAL EM 2019-01-24,2019-01 - Baixas de Construção Concedidas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8544,3100310473202512,REGULARIZACAO,LEVANTAMENTO TOTAL,"RUA TINGUI, 50",DO TIROL,227.0,033,044,203.86,"171,69","171,69",2.0,RESIDENCIAL,1.0,0.0,<NA>,<NA>,19/11/2025,BAIXA TOTAL EM 2025-11-19,2025-11 - Baixas de Construção Concedidas
8545,3100044205202496,REGULARIZACAO,LEVANTAMENTO TOTAL,"RUA PROFESSOR RODRIGO AGNELO ANTUNES, 55",SALGADO FILHO,720.0,057,015,408.0,"334,89","317,51",2.0,RESIDENCIAL,2.0,0.0,<NA>,<NA>,12/11/2025,BAIXA TOTAL EM 2025-11-12,2025-11 - Baixas de Construção Concedidas
8546,3100287999202575,REGULARIZACAO,LEVANTAMENTO DO ACRESCIMO,"RUA JOSE HEMETERIO ANDRADE, 711 / VIA DE PEDES...",ESTORIL,170.0,119,001,1230.0,"2455,84","1612,01",9.0,RESIDENCIAL,16.0,0.0,<NA>,<NA>,12/11/2025,BAIXA TOTAL EM 2025-11-12,2025-11 - Baixas de Construção Concedidas
8547,011005971980,LICENCIAMENTO,MODIF. COM DECRESCIMO DE AREA CONSTRUIDA,"RUA LAVRAS, 732",SEGUNDA SEÇÃO SUBURBANA,102.0,015,005,415.0,"1023,8","580,98",7.0,RESIDENCIAL,9.0,0.0,LUCAS CRISTIANO RODRIGUES CRUZ,A141284-1,10/11/2025,BAIXA TOTAL EM 2025-11-10,2025-11 - Baixas de Construção Concedidas


In [83]:
df_cadastro_imobilario_bhmap = download_data.get_csv_file(url = dicts.DATASETS["BHMAP_DATA"]["cadastro_imobiliario"], separate = ",")

https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:CADASTRO_IMOBILIARIO&srsName=EPSG:31983&outputFormat=csv: successful downloaded data!


In [90]:
df_cadastro_imobilario_bhmap = df_cadastro_imobilario_bhmap\
.assign(split = lambda df: df.index % 3)

In [91]:
df_cadastro_imobilario_bhmap.query("split == 0").to_parquet(path = "../data/raw_cadastro_imobiliario_bhmap_split0.parquet", engine = "fastparquet", compression = "zstd")
df_cadastro_imobilario_bhmap.query("split == 1").to_parquet(path = "../data/raw_cadastro_imobiliario_bhmap_split1.parquet", engine = "fastparquet", compression = "zstd")
df_cadastro_imobilario_bhmap.query("split == 2").to_parquet(path = "../data/raw_cadastro_imobiliario_bhmap_split2.parquet", engine = "fastparquet", compression = "zstd")
df_cadastro_imobilario_bhmap

,fid,id_iptu_ctm,indice_cadastral,nulotctm,zoneamento_pviptu,frequencia_coleta,ind_meio_fio,ind_pavimentacao,ind_arborizacao,ind_galeria_pluvial,ind_iluminacao_publica,ind_rede_esgoto,ind_rede_agua,ind_rede_telefonica,area_terreno,area_construcao,tipo_construtivo,tipo_ocupacao,padrao_acabamento,quantidade_economias,fracao_ideal,tipo_logradouro,nome_logradouro,numero_imovel,cep,zona_homogenia,tipologia,geometria,complemento_endereco,ano_construcao,urlfile,split
0,CADASTRO_IMOBILIARIO.535,535,354008 033A0017,40225200480,ZAR2,COLETA ALTERNADA,SIM,SIM,NÃO,NÃO,SIM,SIM,SIM,SIM,319.00,283.00,CASA,RESIDENCIAL,P2,4,1.000000,RUA,PRIMEIRO DE MAIO,290.0,31130130,NE116,DEMAIS CASOS,MULTIPOLYGON (((610000.247313316 7799916.18066...,NaN,1960,https://geoservicos.pbh.gov.br/geoserver/wfs?s...,0
1,CADASTRO_IMOBILIARIO.536,536,329013 301 5856,140838900900,ZAP,COLETA ALTERNADA,SIM,SIM,SIM,SIM,SIM,SIM,SIM,SIM,69958.44,69.00,APARTAMENTO,RESIDENCIAL,P2,1,0.001042,RUA,CONGONHAL,768.0,31360020,PA307,ALINHAMENTO,MULTIPOLYGON (((604121.587014456 7802685.13705...,BLOCO C APT 304,1982,https://geoservicos.pbh.gov.br/geoserver/wfs?s...,1
2,CADASTRO_IMOBILIARIO.537,537,329013 301 557X,140838900900,ZAP,COLETA ALTERNADA,SIM,SIM,SIM,SIM,SIM,SIM,SIM,SIM,69958.44,59.00,APARTAMENTO,RESIDENCIAL,P2,1,0.000884,RUA,CONGONHAL,728.0,31360020,PA307,ALINHAMENTO,MULTIPOLYGON (((604121.587014456 7802685.13705...,APT 402,1982,https://geoservicos.pbh.gov.br/geoserver/wfs?s...,2
3,CADASTRO_IMOBILIARIO.538,538,006048 302 0211,10277400080,ZCBH,COLETA DIARIA,SIM,SIM,SIM,SIM,SIM,SIM,SIM,SIM,954.00,120.25,APARTAMENTO,RESIDENCIAL,P3,1,0.010280,AVE,AFONSO PENA,1715.0,30130006,CS125,ALINHAMENTO,MULTIPOLYGON (((611670.533779752 7796193.77834...,BLOCO A APT 1102,1968,https://geoservicos.pbh.gov.br/geoserver/wfs?s...,0
4,CADASTRO_IMOBILIARIO.539,539,008009 010 0264,10294800160,ZCBH,COLETA DIARIA,SIM,SIM,SIM,SIM,SIM,SIM,SIM,SIM,1125.00,159.99,APARTAMENTO,RESIDENCIAL,P3,1,0.016082,RUA,OURO PRETO,617.0,30170044,CS101,FRENTE,MULTIPOLYGON (((609856.744885034 7796648.61576...,APT 601,1994,https://geoservicos.pbh.gov.br/geoserver/wfs?s...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893233,CADASTRO_IMOBILIARIO.764994,764994,007003 012B0955,10021800410,ZCBH,COLETA DIARIA,SIM,SIM,NÃO,SIM,SIM,SIM,SIM,SIM,919.00,19.01,VAGA DE GARAGEM NAO RESIDENCIAL,NAO RESIDENCIAL,P5,1,0.003834,RUA,RIO GRANDE DO NORTE,1560.0,30130138,CS210,NaN,MULTIPOLYGON (((611645.886951152 7794902.97108...,GARAGE 23,1988,https://geoservicos.pbh.gov.br/geoserver/wfs?s...,1
893234,CADASTRO_IMOBILIARIO.764995,764995,007004 001A0362,10020500280,ZCBH,COLETA DIARIA,SIM,SIM,SIM,SIM,SIM,SIM,SIM,SIM,654.00,42.71,LOJA,NAO RESIDENCIAL,P2,1,0.021400,AVE,CRISTOVAO COLOMBO,16.0,30140150,CS209,LOJA EM EDIFICIO/GALERIA-FRENTE PARA RUA,MULTIPOLYGON (((611506.301298909 7794916.85826...,LOJA 118,1979,https://geoservicos.pbh.gov.br/geoserver/wfs?s...,2
893235,CADASTRO_IMOBILIARIO.764996,764996,007007 013 0059,10026100335,ZCBH,COLETA DIARIA,SIM,SIM,SIM,SIM,SIM,SIM,SIM,SIM,450.00,232.76,APARTAMENTO,RESIDENCIAL,P4,1,0.083790,RUA,TOME DE SOUZA,235.0,30140130,CS210,FRENTE,MULTIPOLYGON (((612067.538199226 7794933.83480...,APT 601,1988,https://geoservicos.pbh.gov.br/geoserver/wfs?s...,0
893236,CADASTRO_IMOBILIARIO.764997,764997,008024 001 2127,10364800015,ZCBH,COLETA DIARIA,SIM,SIM,SIM,SIM,SIM,SIM,SIM,SIM,860.00,22.88,VAGA DE GARAGEM NAO RESIDENCIAL,NAO RESIDENCIAL,P3,1,0.002266,AVE,AUGUSTO DE LIMA,1376.0,30190003,CS129,NaN,MULTIPOLYGON (((609998.499194326 7796873.10007...,GARAGE 18 1 SUBSL,1995,https://geoservicos.pbh.gov.br/geoserver/wfs?s...,1


In [ ]:
df_enderecamento_bhmap = download_data.get_csv_file(url = dicts.DATASETS["BHMAP_DATA"]["enderecamento"], separate = ",")

https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:ENDERECO&srsName=EPSG:31983&outputFormat=csv: successful downloaded data!


In [25]:
df_enderecamento_bhmap.to_parquet(path = "../data/raw_enderecamento_bhmap.parquet", engine = "fastparquet", compression = "zstd")
df_enderecamento_bhmap

,fid,idend,id_edc,id_logradouro,sigla_tipo_logradouro,desc_tipo_logradouro,nome_logradouro,numero_imovel,letra_imovel,id_bairro_popular,num_bairro_popular,nome_bairro_popular,id_bairro_oficial,num_bairro_oficial,tipo_bairro_oficial,nome_bairro_oficial,id_regional,nome_regional,cep,existencia_num_local,situacao_pbh,geometria,urlfile
0,ENDERECO.539435,12406200267,539435,124062,RUA,RUA,CASTELO DE ALCOBACA,267,NaN,34,639,Castelo,80,295,Bairro,do Castelo,8,PAMPULHA,31330040.0,Sim,Número Oficial,POINT (605126.039304405 7801166.85898991),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
1,ENDERECO.963552,02589800275,963552,25898,RUA,RUA,JOSE DE PAULA COTTA,275,NaN,181,806,Tupi A,41,967,Bairro,Tupi,6,NORTE,31842080.0,a confimar,Número Oficial,POINT (612301.407745009 7806273.51170476),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
2,ENDERECO.23642,03209300380,23642,32093,RUA,RUA,GUANHAES,380,NaN,41,646,Colégio Batista,302,106,Seção Suburbana,Sexta,3,LESTE,31110160.0,Sim,Número Oficial,POINT (611343.217321981 7798471.34177438),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
3,ENDERECO.983010,05620300040,983010,56203,RUA,RUA,LUZIA CIRILA RODRIGUES,40,NaN,161,785,São João Batista,273,908,Bairro,São João Batista,9,VENDA NOVA,31520210.0,a confimar,Número Oficial,POINT (607773.90993111 7808293.4808703),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
4,ENDERECO.91185,04399500170,91185,43995,RUA,RUA,MARIA EUZEBIA,170,NaN,113,731,Nova Gameleira,158,244,Bairro,da Gameleira,7,OESTE,30510360.0,Sim,Número Oficial,POINT (605600.945644327 7794356.81622861),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755825,ENDERECO.723800,01350400457A,723800,13504,RUA,RUA,FLOR DE VIDRO,457,A,249,1172,Jardim Alvorada,231,809,Bairro,dos Manacás,8,PAMPULHA,30810330.0,Sim,Número Inválido,POINT (605580.989599566 7799534.14622632),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
755826,ENDERECO.544348,09966500020,544348,99665,RUA,RUA,FLOR DA PAPOULA,20,NaN,249,1172,Jardim Alvorada,231,809,Bairro,dos Manacás,8,PAMPULHA,30810720.0,Não,Número Oficial,POINT (605541.696707875 7799530.98385999),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
755827,ENDERECO.746491,09901600140,746491,99016,RUA,RUA,RADIALISTA JOSUE POLICARPO,140,NaN,38,643,Céu Azul,12,988,Bairro,Céu Azul,9,VENDA NOVA,31578430.0,Sim,NaN,POINT (604117.348187935 7808991.05563589),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
755828,ENDERECO.418759,01350400435,418759,13504,RUA,RUA,FLOR DE VIDRO,435,NaN,249,1172,Jardim Alvorada,231,809,Bairro,dos Manacás,8,PAMPULHA,30810330.0,Não,Número Oficial,POINT (605581.123299481 7799498.07727319),https://geoservicos.pbh.gov.br/geoserver/wfs?s...


In [ ]:
df_logradouros_bhmap = download_data.get_csv_file(url = dicts.DATASETS["BHMAP_DATA"]["logradouros"], separate = ",")

https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:LOGRADOURO&srsName=EPSG:31983&outputFormat=csv: successful downloaded data!


In [28]:
df_logradouros_bhmap.to_parquet(path = "../data/raw_logradouros_bhmap.parquet", engine = "fastparquet", compression = "zstd")
df_logradouros_bhmap

,fid,id_logradouro,cod_logradouro,tipo_logradouro,nome_logradouro,largura_media,comprimento_logradouro,geometria,urlfile
0,LOGRADOURO.2251,2251,2251,RUA,ALFREDINA AMARAL,11.63,1181.30,MULTILINESTRING ((604124.119988358 7790735.207...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
1,LOGRADOURO.2264,2264,2264,AVENIDA,PROFESSOR ALFREDO BALENA,32.80,794.12,MULTILINESTRING ((611896.250430743 7796329.797...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
2,LOGRADOURO.2277,2277,2277,AVENIDA,ALFREDO CAMARATE,17.57,1135.26,MULTILINESTRING ((607212.578230775 7803369.982...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
3,LOGRADOURO.2311,2311,2311,RUA,ALIANCA,6.11,120.53,MULTILINESTRING ((605041.89745049 7797231.8470...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
4,LOGRADOURO.2324,2324,2324,RUA,ALIANCA,11.90,198.49,MULTILINESTRING ((613318.857094893 7804134.440...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
...,...,...,...,...,...,...,...,...,...
16525,LOGRADOURO.313943,313943,313943,BECO,CINCO MIL CENTO E QUATORZE,2.00,21.73,MULTILINESTRING ((614351.20828531 7806095.9827...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
16526,LOGRADOURO.313944,313944,313944,BECO,CINCO MIL CENTO E QUINZE,2.00,48.64,MULTILINESTRING ((614234.69524123 7805678.3942...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
16527,LOGRADOURO.313945,313945,313945,BECO,CINCO MIL CENTO E DEZESSEIS,1.30,27.08,MULTILINESTRING ((614019.148259312 7805599.723...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
16528,LOGRADOURO.313947,313947,313947,BECO,SEM NOME,1.50,36.79,MULTILINESTRING ((606022.839167404 7800345.842...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...


In [ ]:
df_atividades_economicas_bhmap = download_data.get_csv_file(url = dicts.DATASETS["BHMAP_DATA"]["atividade_economica"], separate = ",")

https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:ATIVIDADE_ECONOMICA&srsName=EPSG:31983&outputFormat=csv: successful downloaded data!


In [31]:
df_atividades_economicas_bhmap.to_parquet(path = "../data/raw_atividades_economicas_bhmap.parquet", engine = "fastparquet", compression = "zstd")
df_atividades_economicas_bhmap

,fid,id_ativ_econ_estabelecimento,cnae_principal,descricao_cnae_principal,cnae,data_inicio_atividade,natureza_juridica,porte_empresa,area_utilizada,ind_simples,ind_mei,ind_possui_alvara,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,nome,nome_fantasia,cnpj,geometria,urlfile
0,ATIVIDADE_ECONOMICA.1023,1023,4744005.0,COMERCIO VAREJISTA DE MATERIAIS DE CONSTRUCAO ...,"4742300, 4744005",19/05/2021,SOCIEDADE EMPRESÁRIA LIMITADA,DEMAIS,11.0,N,N,SIM,UNIDADE PRODUTIVA,INTERNET,RUA,ESTORIL,1860,SALA A,SAO FRANCISCO,COMERCIAL DE BOMBAS E MOTORES LTDA,PARAISO DAS BOMBAS ON LINE,57359000448,POINT (608657.55845935 7801386.07496236),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
1,ATIVIDADE_ECONOMICA.1024,1024,4649408.0,"COMERCIO ATACADISTA DE PRODUTOS DE HIGIENE, L...","4649408, 4679604",16/03/2023,SOCIEDADE EMPRESÁRIA LIMITADA,DEMAIS,400.0,N,N,SIM,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,ESTORIL,1860,NaN,SAO FRANCISCO,COMERCIAL DE BOMBAS E MOTORES LTDA,CD ULTRACLOR,57359000529,POINT (608657.55845935 7801386.07496236),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
2,ATIVIDADE_ECONOMICA.1025,1025,6203100.0,DESENVOLVIMENTO E LICENCIAMENTO DE PROGRAMAS D...,"6202300, 6203100, 6204000",05/02/1994,SOCIEDADE EMPRESÁRIA LIMITADA,EMPRESA DE PEQUENO PORTE,10.0,S,N,SIM,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,DOS GUAJAJARAS,910,SALA 620,CENTRO,I2 INTERCAMBIO DE INFORMACOES SOFTWARE LTDA,NaN,86370285000178,POINT (610658.099749675 7796519.69800442),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
3,ATIVIDADE_ECONOMICA.1026,1026,4723700.0,COMERCIO VAREJISTA DE BEBIDAS,"4723700, 4723700, 4930201, 4930202, 5212500",01/07/1993,SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,18.0,S,N,SIM,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,CELIO DE CASTRO,869,NaN,FLORESTA,REFRIGERANTES CANAL LTDA,TELE BEBIDA,71140008000100,POINT (611887.702388613 7797823.01100151),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
4,ATIVIDADE_ECONOMICA.1027,1027,6821802.0,CORRETAGEM NO ALUGUEL DE IMOVEIS,"6821801, 6821802",02/05/1994,SOCIEDADE SIMPLES LIMITADA,EMPRESA DE PEQUENO PORTE,30.0,S,N,NÃO,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,ERICO VERISSIMO,1017,NaN,SANTA MONICA,IMOVEIS J & M LTDA,NaN,72877000198,POINT (607659.026513834 7808690.81916982),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
550795,ATIVIDADE_ECONOMICA.499315,499315,8599604.0,TREINAMENTO EM DESENVOLVIMENTO PROFISSIONAL E ...,"6204000, 8599604",27/05/2025,SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,10.0,S,N,NÃO,UNIDADE PRODUTIVA,ATIVIDADES DESENVOLVIDAS FORA DO ESTABELECIMEN...,ALA,TOCARI,170,NaN,DOM CABRAL,NEMO CONSULTING LTDA,NEMO CONSULTING,61029601000111,POINT (604866.909533846 7796812.68772574),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
550796,ATIVIDADE_ECONOMICA.499316,499316,9602501.0,"CABELEIREIROS, MANICURE E PEDICURE",9602501,27/05/2025,EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,10.0,S,S,NÃO,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,GOUVEIA,80,NaN,VILA SAO GERALDO,32.421.289 ANA DARA DA SILVA CRISTIANO,NaN,32421289000109,POINT (615243.87710741 7799660.27168474),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
550797,ATIVIDADE_ECONOMICA.499317,499317,7311400.0,AGENCIAS DE PUBLICIDADE,"7311400, 7319002, 7319003, 7319004",27/05/2025,SOCIEDADE ANÔNIMA FECHADA,DEMAIS,7.0,N,N,SIM,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVE,RAJA GABAGLIA,3502,SALA: 201,ESTORIL,HIGH STAKES S.A,NaN,22123183000256,POINT (609236.038955649 7792199.05146436),https://geoservicos.pbh.gov.br/geoserver/wfs?s...
550798,ATIVIDADE_ECONOMICA.499318,499318,7911200.0,AGENCIAS DE VIAGENS,"6311900, 7490104, 7911200, 7912100, 7990200, 8...",27/05/2025,SOCIEDADE EMPRESÁRIA LIMITADA,EMPRESA DE PEQUENO PORTE,5.0,S,N,NÃO,UNIDADE PRODUTIVA,ATIVIDADES DESENVOLVIDAS FORA DO ESTABELECIMEN...,AVE,SARAMENHA,1826,"BLOCO: 19 ,APT: 401",GUARANI,DUART_MILES LTDA,NaN,51983208000109,POINT

In [ ]:
df_atividades_economicas_autonomos_bhmap = download_data.get_csv_file(url = dicts.DATASETS["BHMAP_DATA"]["atividade_economica_autonomos"], separate = ",")

https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:ATIVIDADE_ECONOMICAS_AUTONOMOS&srsName=EPSG:31983&outputFormat=csv: successful downloaded data!


In [34]:
df_atividades_economicas_autonomos_bhmap.to_parquet(path = "../data/raw_atividades_economicas_autonomos_bhmap.parquet", engine = "fastparquet", compression = "zstd")
df_atividades_economicas_autonomos_bhmap

,fid,cod_inscricao_municipal,area_utilizada,area_sujeita_tfs,codigo_cbo,descricao_cbo,tipo_logradouro,logradouro,numero,complemento,geometria,data_inicio_atividade,nome_bairro_popular,nome_regional,urlfile
0,ATIVIDADE_ECONOMICAS_AUTONOMOS.06464160013,06464160013,10,10.0,223208,CIRURGIAO DENTISTA - CLINICO GERAL,RUA,TOME DE SOUZA,215.0,APTO 101,POINT (612084.154050344 7794959.8960433),05/09/2002,Savassi,CENTRO-SUL,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
1,ATIVIDADE_ECONOMICAS_AUTONOMOS.11835050012,11835050012,10,NaN,333110,INSTRUTOR DE CURSOS LIVRES,RUA,TOME DE SOUZA,248.0,NaN,POINT (612079.968980834 7794982.41069816),16/10/2019,Savassi,CENTRO-SUL,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
2,ATIVIDADE_ECONOMICAS_AUTONOMOS.05714730018,05714730018,10,10.0,225250,MÉDICO GINECOLOGISTA E OBSTETRA,RUA,TOME DE SOUZA,248.0,APT 500,POINT (612079.968980834 7794982.41069816),13/01/1987,Savassi,CENTRO-SUL,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
3,ATIVIDADE_ECONOMICAS_AUTONOMOS.05922110019,05922110019,17,17.0,225125,MEDICO CLINICO,AVENIDA,DO CONTORNO,5326.0,CASA - SALA 01,POINT (612127.557478547 7794886.59225642),30/07/1992,Savassi,CENTRO-SUL,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
4,ATIVIDADE_ECONOMICAS_AUTONOMOS.05902630017,05902630017,17,17.0,225125,MEDICO CLINICO,AVENIDA,DO CONTORNO,5326.0,"CASA , SALA 1",POINT (612127.557478547 7794886.59225642),01/01/1992,Savassi,CENTRO-SUL,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23942,ATIVIDADE_ECONOMICAS_AUTONOMOS.15475710019,15475710019,16,16.0,223208,CIRURGIAO DENTISTA - CLINICO GERAL,RUA,DOS GOITACAZES,71.0,SALA 802,POINT (611162.173341908 7796653.9468948),22/03/2024,Centro,HIPERCENTRO,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
23943,ATIVIDADE_ECONOMICAS_AUTONOMOS.05668460017,05668460017,46,46.0,225125,MEDICO CLINICO,RUA,DOS GOITACAZES,71.0,SALA 1408,POINT (611162.173341908 7796653.9468948),18/03/1986,Centro,HIPERCENTRO,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
23944,ATIVIDADE_ECONOMICAS_AUTONOMOS.05589340019,05589340019,40,40.0,223208,CIRURGIAO DENTISTA - CLINICO GERAL,RUA,DOS GOITACAZES,71.0,SALA 905,POINT (611162.173341908 7796653.9468948),01/10/1983,Centro,HIPERCENTRO,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
23945,ATIVIDADE_ECONOMICAS_AUTONOMOS.05641770015,05641770015,10,10.0,223208,CIRURGIAO DENTISTA - CLINICO GERAL,RUA,DOS GOITACAZES,71.0,SALA 611,POINT (611162.173341908 7796653.9468948),10/01/1985,Centro,HIPERCENTRO,https://geoservicos.pbh.gov.br/geoserver/wfs?s...


In [ ]:
df_conjuntos_habitacionais_bhmap = download_data.get_csv_file(url = dicts.DATASETS["BHMAP_DATA"]["conjuntos_habitacionais"], separate = ",")

https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:CONJUNTO_HABITACIONAL&srsName=EPSG:31983&outputFormat=csv: successful downloaded data!


In [37]:
df_conjuntos_habitacionais_bhmap.to_parquet(path = "../data/raw_conjuntos_habitacionais_bhmap.parquet", engine = "fastparquet", compression = "zstd")
df_conjuntos_habitacionais_bhmap

,fid,id_chb,nome_conjunto_habitacional,desc_agrupamento,status_construcao,status_regularizacao,num_unid_iniciar,num_unid_andamento,num_unid_conclusao,desc_programa,desc_fonte_recurso,ano_previsao_conclusao,ano_primeira_entrega_uh,ano_ultima_entrega_uh,desc_faixa_renda,geometria,urlfile
0,CONJUNTO_HABITACIONAL.255,255,Conjunto Habitacional Lagoa- R4 (Lagoa),376 UH - Lagoa,Obra concluída,Regularizado,NaN,NaN,17.0,Acampados / OPH / PEAR -> Desabrigados,Pró-moradia,NaN,1999.0,1999.0,NaN,MULTIPOLYGON (((604796.459895016 7809010.76948...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
1,CONJUNTO_HABITACIONAL.374,374,Roseiral (São João E31),296 UH - São João,Obra concluída,Em andamento,NaN,NaN,8.0,Vila Viva,BNDES / PAC 1 / Saneamento Para Todos,NaN,2007.0,2007.0,NaN,MULTIPOLYGON (((614580.369604094 7795330.86500...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
2,CONJUNTO_HABITACIONAL.257,257,Conjunto Habitacional Lagoa- R6 (Lagoa),376 UH - Lagoa,Obra concluída,Regularizado,NaN,NaN,8.0,Acampados / OPH / PEAR -> Desabrigados,Pró-moradia,NaN,1999.0,1999.0,NaN,MULTIPOLYGON (((604785.87223094 7809261.718408...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
3,CONJUNTO_HABITACIONAL.262,262,Conjunto Habitacional Lagoa- R11 (Lagoa),376 UH - Lagoa,Obra concluída,Regularizado,NaN,NaN,44.0,Acampados / OPH / PEAR -> Desabrigados,Pró-moradia,NaN,1999.0,1999.0,NaN,MULTIPOLYGON (((604819.226671117 7809000.18164...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
4,CONJUNTO_HABITACIONAL.921,921,Conjunto Nova Cachoeirinha,NaN,Projeto concluído,NaN,48.0,NaN,NaN,OP,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON (((609689.082132964 7800919.10652...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,CONJUNTO_HABITACIONAL.175,175,Residencial Quaresmeira,440 UH - Vila Viva São Tomaz,Obra concluída,Não regularizado,NaN,NaN,32.0,Vila Viva,FMS - Fundo Municipal de Saneamento / PAC 1,NaN,2020.0,2020.0,NaN,MULTIPOLYGON (((609835.577802726 7805199.00536...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
463,CONJUNTO_HABITACIONAL.274,274,Residencial Paineira,440 UH - Vila Viva São Tomaz,Obra concluída,Regularizado,NaN,NaN,24.0,Vila Viva,FMS - Fundo Municipal de Saneamento / PAC 1,NaN,2023.0,2023.0,NaN,MULTIPOLYGON (((609884.731280574 7805225.68993...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
464,CONJUNTO_HABITACIONAL.821,821,Residencial Flor de Maio (Conjunto 2 - Ar Prin...,44 UH - AR Principal,Obra concluída,Não regularizado,NaN,NaN,10.0,Vila Viva,FMS - Fundo Municipal de Saneamento / PAC 1,2021.0,2021.0,2021.0,NaN,MULTIPOLYGON (((610658.006956364 7792923.59834...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
465,CONJUNTO_HABITACIONAL.276,276,Residencial Buritizeiro,440 UH - Vila Viva São Tomaz,Obra concluída,Não regularizado,32.0,NaN,32.0,Vila Viva,FMS - Fundo Municipal de Saneamento / PAC 1,NaN,2023.0,2023.0,NaN,MULTIPOLYGON (((610196.97666653 7805120.931253...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...


In [ ]:
df_lotes_ctm = download_data.get_csv_file(url = dicts.DATASETS["BHMAP_DATA"]["lote_ctm"], separate = ",")

https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:LOTE_CTM&srsName=EPSG:31983&outputFormat=csv: successful downloaded data!


In [40]:
df_lotes_ctm.to_parquet(path = "../data/raw_lotes_ctm_bhmap.parquet", engine = "fastparquet", compression = "zstd")
df_lotes_ctm

,fid,id_lt,nulotctm,id_quadra_ctm,area_m2,geometria,urlfile
0,LOTE_CTM.841616,841616,210283600520,14004,131.71,MULTIPOLYGON (((604857.833714111 7807573.72199...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
1,LOTE_CTM.849063,849063,210287700310,14171,305.17,MULTIPOLYGON (((604883.80654229 7807928.769636...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
2,LOTE_CTM.864246,864246,200918000300,28623,472.21,MULTIPOLYGON (((606381.549951401 7806197.91219...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
3,LOTE_CTM.866309,866309,210285100210,14173,171.34,MULTIPOLYGON (((604630.35560853 7807789.348155...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
4,LOTE_CTM.870112,870112,210287700075,14171,204.13,MULTIPOLYGON (((604867.82767841 7807828.880108...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
...,...,...,...,...,...,...,...
365388,LOTE_CTM.556259,556259,40509000295,2467,352.81,MULTIPOLYGON (((610799.549984302 7801432.87194...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
365389,LOTE_CTM.557879,557879,40538000040,2436,368.42,MULTIPOLYGON (((610846.155496263 7801500.79812...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
365390,LOTE_CTM.546803,546803,40538000050,2436,379.46,MULTIPOLYGON (((610831.387189129 7801501.53797...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
365391,LOTE_CTM.546806,546806,40538000115,2436,370.01,MULTIPOLYGON (((610848.126253049 7801536.71081...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...


In [ ]:
df_projetos_edificacao_licenciados_bhmap = download_data.get_csv_file(url = dicts.DATASETS["BHMAP_DATA"]["projeto_edificacao_licenciado"], separate = ",")

https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:PROJETO_EDIFICACAO_LICENCIADO&srsName=EPSG:31983&outputFormat=csv: successful downloaded data!


In [ ]:
df_projetos_edificacao_licenciados_bhmap.to_parquet(path = "../data/raw_projetos_edificacao_licenciados_bhmap.parquet", engine = "fastparquet", compression = "zstd")
df_projetos_edificacao_licenciados_bhmap

,fid,id_projeto_edificacoes,numero_processo,num_requerimento,situacao_requerimento,titulo_projeto,tipo,situacao_projeto,num_ultimo_alvara,dt_emissao_alvara_construcao,dt_concessao_ultimo_alvara,dt_validade_ultimo_alvara,data_comunicado_inicio_obra,data_ultima_baixa,tipo_ultima_baixa,endereco,lote_projeto,uso_geral,qtd_und_residencial,qtd_und_nao_residencial,area_construida,tipo_aprovacao,data_aprovacao,area_liquida,qtde_pavimentos,link_siatu_edificacao,geometria,urlfile
0,PROJETO_EDIFICACAO_LICENCIADO.80842,80842,1.000005e+10,1997M00354,Deferido,APROVACAO INICIAL,LICENCIAMENTO,APROVADO,199900003.0,2025-09-09,1998-12-30,2002-12-30,NaN,2002-01-28,BAIXA TOTAL,"RUA DAS AMOREIRAS, 71, 81, 93;RUA DOS MAMOEIRO...","Zona Fiscal 948 Quarteirão 006 Lote(s) 014,01...",RESIDENCIAL,120,0,5444.00,NaN,30/12/1998,4773.39,40,"<a target=""_blank"" href=""https://urbano.pbh.go...",MULTIPOLYGON (((609963.580647039 7807967.19127...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
1,PROJETO_EDIFICACAO_LICENCIADO.299334,299334,1.000025e+10,2024A22036,Deferido,APROVACAO INICIAL,LICENCIAMENTO,APROVADO,199002457.0,2024-07-31,1990-12-26,1991-12-26,NaN,NaN,NaN,NaN,Zona Fiscal 725 Quarteirão 013 Lote(s) 001,NÃO RESIDENCIAL,0,1,26.82,Convencional,26/12/1990,0.00,1,"<a target=""_blank"" href=""https://urbano.pbh.go...",MULTIPOLYGON (((603478.872405817 7801584.37717...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
2,PROJETO_EDIFICACAO_LICENCIADO.125367,125367,1.000061e+10,1994M01581,Deferido,APROVACAO INICIAL,LICENCIAMENTO,APROVADO,199500304.0,1995-01-11,1995-01-02,2006-12-02,NaN,NaN,NaN,"RUA DOM LARA, 130",Zona Fiscal 474 Quarteirão 044 Lote(s) 019,USO INVÁLIDO: PROJETO MIGRADO EM 2013,0,0,0.00,NaN,02/01/1995,394.22,2,"<a target=""_blank"" href=""https://urbano.pbh.go...",MULTIPOLYGON (((604924.936291498 7792977.62281...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
3,PROJETO_EDIFICACAO_LICENCIADO.80125,80125,1.000104e+10,1996M00723,Deferido,MODIFICACAO COM ACRESCIMO DE AREA CONSTRUIDA,LICENCIAMENTO,APROVADO,199600023.0,1996-01-03,1995-12-28,1997-06-28,NaN,NaN,NaN,"RUA ANTONIO CAMPOS, 168",Zona Fiscal 244 Quarteirão 104 Lote(s) 049,RESIDENCIAL,0,0,0.00,NaN,28/12/1995,259.55,4,"<a target=""_blank"" href=""https://urbano.pbh.go...",MULTIPOLYGON (((605582.232814629 7794590.81298...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
4,PROJETO_EDIFICACAO_LICENCIADO.97099,97099,1.003519e+10,2007M01763,Deferido,MODIFICACAO COM ACRESCIMO DE AREA CONSTRUIDA,LICENCIAMENTO,APROVADO,NaN,NaN,NaN,NaN,NaN,2007-06-26,BAIXA TOTAL,NaN,Zona Fiscal 012 Quarteirão 022 Lote(s) 024,NÃO RESIDENCIAL,0,41,2314.15,Convencional,05/03/1999,1352.07,11,"<a target=""_blank"" href=""https://urbano.pbh.go...",MULTIPOLYGON (((609288.100434275 7795992.28469...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82617,PROJETO_EDIFICACAO_LICENCIADO.320448,320448,NaN,2025A27430,Deferido,APROVACAO INICIAL,LICENCIAMENTO,APROVADO,194900352.0,2025-08-29,1949-03-23,1950-03-23,NaN,1950-06-01,BAIXA TOTAL,NaN,Zona Fiscal 130 Quarteirão 107A Lote(s) 005,RESIDENCIAL,1,0,67.00,Convencional,23/03/1949,0.00,1,"<a target=""_blank"" href=""https://urbano.pbh.go...",MULTIPOLYGON (((613414.200014578 7797613.10944...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
82618,PROJETO_EDIFICACAO_LICENCIADO.320451,320451,NaN,2025A27433,Deferido,APROVACAO INICIAL,LICENCIAMENTO,APROVADO,197300895.0,2025-08-31,1973-07-19,1973-07-19,NaN,1973-07-19,BAIXA TOTAL,NaN,Zona Fiscal 104 Quarteirão 122A Lote(s) 023B,RESIDENCIAL,12,0,2080.00,Convencional,19/07/1973,0.00,5,"<a target=""_blank"" href=""https://urbano.pbh.go...",MULTIPOLYGON (((608770.404374105 7795846.63900...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
82619,PROJETO_EDIFICACAO_LICENCIADO.321150,321150,NaN,2025A27953,Deferido,APROVACAO INICIAL,LICENCIAMENTO,APROVADO,196600144.0,2025-10-07,1966-02-16,1967-02-16,NaN,1967-11-07,BAIXA TOTAL,NaN,Z

In [ ]:
df_zona_homogenea_iptu_bhmap = download_data.get_csv_file(url = dicts.DATASETS["BHMAP_DATA"]["zona_homogenea_iptu"], separate = ",")

https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:ZONA_HOMOGENEA_IPTU&srsName=EPSG:31983&outputFormat=csv: successful downloaded data!


In [53]:
df_zona_homogenea_iptu_bhmap.to_parquet(path = "../data/raw_zona_homogenea_iptu_bhmap.parquet", engine = "fastparquet", compression = "zstd")
df_zona_homogenea_iptu_bhmap

,fid,id_zona_homogenea,codigo_zh,geometria,urlfile
0,ZONA_HOMOGENEA_IPTU.1423,1423,LE131,MULTIPOLYGON (((616396.845584095 7800067.77980...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
1,ZONA_HOMOGENEA_IPTU.1426,1426,LE108,MULTIPOLYGON (((613580.3455313 7799411.2434438...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
2,ZONA_HOMOGENEA_IPTU.1428,1428,PA343,MULTIPOLYGON (((604809.991325505 7799512.17289...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
3,ZONA_HOMOGENEA_IPTU.1324,1324,NE109,MULTIPOLYGON (((609705.81295104 7801580.331932...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
4,ZONA_HOMOGENEA_IPTU.1405,1405,LE104,MULTIPOLYGON (((612429.17619251 7799863.261436...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
...,...,...,...,...,...
981,ZONA_HOMOGENEA_IPTU.1925,1925,NO240,MULTIPOLYGON (((607824.548312448 7798010.35858...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
982,ZONA_HOMOGENEA_IPTU.1138,1138,PA434,MULTIPOLYGON (((609352.016755677 7806443.36207...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
983,ZONA_HOMOGENEA_IPTU.1140,1140,NT216,MULTIPOLYGON (((613379.95502322 7806518.276827...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...
984,ZONA_HOMOGENEA_IPTU.1882,1882,BA122,MULTIPOLYGON (((600455.56709891 7789100.292208...,https://geoservicos.pbh.gov.br/geoserver/wfs?s...


In [23]:
dicts.bhmap_data

{'cadastro_imobiliario': 'https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:CADASTRO_IMOBILIARIO&srsName=EPSG:31983&outputFormat=csv',
 'atividade_economica': 'https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:ATIVIDADE_ECONOMICA&srsName=EPSG:31983&outputFormat=csv',
 'atividade_economica_autonomos': 'https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:ATIVIDADE_ECONOMICAS_AUTONOMOS&srsName=EPSG:31983&outputFormat=csv',
 'conjuntos_habitacionais': 'https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:CONJUNTO_HABITACIONAL&srsName=EPSG:31983&outputFormat=csv',
 'enderecamento': 'https://geoservicos.pbh.gov.br/geoserver/wfs?service=WFS&version=1.0.0&request=GetFeature&typeName=ide_bhgeo:ENDERECO&srsName=EPSG:31983&outputFormat=csv',
 'logradouros': 'https://geo